# TP 06: Comparative Study of Clustering Algorithms on Real-World Data
## Fundamentals of Artificial Intelligence

**Objective**: Compare K-means clustering and hierarchical clustering on a real-world dataset.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 2. Dataset Selection and Loading

We'll use the **Iris Dataset** from the UCI Machine Learning Repository.

**Justification**: 
- Contains 4 numeric features (sepal length, sepal width, petal length, petal width)
- 150 instances (iris flower samples from 3 species)
- Classic dataset for clustering to identify different flower species based on physical measurements
- No categorical variables, making it ideal for distance-based clustering
- Well-known structure with natural groupings

In [ ]:
# Load the Iris dataset
from sklearn.datasets import load_iris

iris = load_iris()
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['species'] = iris.target

print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
df.head()

In [ ]:
# Dataset information
print("Dataset Info:")
df.info()
print("\nStatistical Summary:")
df.describe()

## 3. Data Preprocessing

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
print("\nTotal missing values:", df.isnull().sum().sum())

In [ ]:
# Select key features for clustering (excluding the species label)
# We'll use all 4 features for iris flowers
features_to_use = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']

# Create feature matrix
X = df[features_to_use].values

print(f"Selected features: {features_to_use}")
print(f"Feature matrix shape: {X.shape}")

In [ ]:
# Feature scaling using standardization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled features - Mean:")
print(X_scaled.mean(axis=0))
print("\nScaled features - Std:")
print(X_scaled.std(axis=0))

In [ ]:
# Visualize feature distributions before and after scaling
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, feature in enumerate(features_to_use):
    axes[0, i].hist(X[:, i], bins=30, edgecolor='black', alpha=0.7)
    axes[0, i].set_title(f'{feature}\n(Original)')
    axes[0, i].set_ylabel('Frequency')
    
    axes[1, i].hist(X_scaled[:, i], bins=30, edgecolor='black', alpha=0.7)
    axes[1, i].set_title(f'{feature}\n(Scaled)')
    axes[1, i].set_ylabel('Frequency')

plt.tight_layout()
plt.suptitle('Feature Distributions: Original vs Scaled', y=1.02, fontsize=14, fontweight='bold')
plt.show()

## 4. K-means Clustering

### 4.1 Elbow Method

In [ ]:
# Apply K-means for different values of K
K_range = range(2, 11)
inertias = []
silhouette_scores = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

print("K-means evaluation completed for K = 2 to 10")

In [ ]:
# Plot Elbow Method and Silhouette Scores
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Elbow plot
ax1.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Number of Clusters (K)', fontsize=12)
ax1.set_ylabel('Within-Cluster Sum of Squares (Inertia)', fontsize=12)
ax1.set_title('Elbow Method', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Silhouette score plot
ax2.plot(K_range, silhouette_scores, 'ro-', linewidth=2, markersize=8)
ax2.set_xlabel('Number of Clusters (K)', fontsize=12)
ax2.set_ylabel('Silhouette Score', fontsize=12)
ax2.set_title('Silhouette Score Analysis', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print silhouette scores
print("\nSilhouette Scores for different K values:")
for k, score in zip(K_range, silhouette_scores):
    print(f"K = {k}: {score:.4f}")

### 4.2 Apply K-means with Optimal K

In [ ]:
# Based on elbow and silhouette analysis, choose optimal K
optimal_k = 3  # This can be adjusted based on the plots above

kmeans_optimal = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
kmeans_labels = kmeans_optimal.fit_predict(X_scaled)

print(f"Optimal K selected: {optimal_k}")
print(f"Silhouette Score: {silhouette_score(X_scaled, kmeans_labels):.4f}")
print(f"\nCluster distribution:")
unique, counts = np.unique(kmeans_labels, return_counts=True)
for cluster, count in zip(unique, counts):
    print(f"Cluster {cluster}: {count} samples ({count/len(kmeans_labels)*100:.1f}%)")

### 4.3 Visualize K-means Clusters

In [ ]:
# 2D visualization using first two features
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

# Plot all feature pairs
plot_idx = 0
for i in range(len(features_to_use)):
    for j in range(i+1, len(features_to_use)):
        if plot_idx < 6:
            scatter = axes[plot_idx].scatter(X_scaled[:, i], X_scaled[:, j], 
                                            c=kmeans_labels, cmap='viridis', 
                                            alpha=0.6, s=50, edgecolors='k', linewidth=0.5)
            axes[plot_idx].set_xlabel(features_to_use[i], fontsize=10)
            axes[plot_idx].set_ylabel(features_to_use[j], fontsize=10)
            axes[plot_idx].set_title(f'K-means Clusters\n{features_to_use[i]} vs {features_to_use[j]}', 
                                    fontsize=11, fontweight='bold')
            axes[plot_idx].grid(True, alpha=0.3)
            plot_idx += 1

plt.colorbar(scatter, ax=axes, label='Cluster')
plt.tight_layout()
plt.show()

## 5. Hierarchical Clustering

### 5.1 Compute Linkage and Plot Dendrograms

In [ ]:
# Perform hierarchical clustering with different linkage methods
linkage_methods = ['single', 'complete', 'average', 'ward']
linkage_matrices = {}

for method in linkage_methods:
    linkage_matrices[method] = linkage(X_scaled, method=method)
    print(f"{method.capitalize()} linkage computed")

In [ ]:
# Plot dendrograms for different linkage methods
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for idx, method in enumerate(linkage_methods):
    dendrogram(linkage_matrices[method], ax=axes[idx], truncate_mode='lastp', p=30)
    axes[idx].set_title(f'Dendrogram - {method.capitalize()} Linkage', 
                       fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('Sample Index or (Cluster Size)', fontsize=11)
    axes[idx].set_ylabel('Distance', fontsize=11)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 5.2 Determine Optimal Number of Clusters

In [ ]:
# Evaluate hierarchical clustering for different numbers of clusters
n_clusters_range = range(2, 11)
hierarchical_scores = {method: [] for method in linkage_methods}

for method in linkage_methods:
    for n_clusters in n_clusters_range:
        hier_model = AgglomerativeClustering(n_clusters=n_clusters, linkage=method)
        labels = hier_model.fit_predict(X_scaled)
        score = silhouette_score(X_scaled, labels)
        hierarchical_scores[method].append(score)

print("Hierarchical clustering evaluation completed")

In [ ]:
# Plot silhouette scores for different linkage methods
plt.figure(figsize=(12, 6))

for method in linkage_methods:
    plt.plot(n_clusters_range, hierarchical_scores[method], 'o-', 
            label=f'{method.capitalize()} Linkage', linewidth=2, markersize=8)

plt.xlabel('Number of Clusters', fontsize=12)
plt.ylabel('Silhouette Score', fontsize=12)
plt.title('Hierarchical Clustering: Silhouette Score Comparison', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print best scores for each method
print("\nBest Silhouette Scores:")
for method in linkage_methods:
    best_idx = np.argmax(hierarchical_scores[method])
    best_k = list(n_clusters_range)[best_idx]
    best_score = hierarchical_scores[method][best_idx]
    print(f"{method.capitalize()} Linkage: K = {best_k}, Score = {best_score:.4f}")

### 5.3 Apply Hierarchical Clustering with Optimal Parameters

In [ ]:
# Choose the best linkage method and number of clusters
best_linkage = 'ward'  # Usually performs well
optimal_hier_k = 3

hier_model = AgglomerativeClustering(n_clusters=optimal_hier_k, linkage=best_linkage)
hier_labels = hier_model.fit_predict(X_scaled)

print(f"Hierarchical Clustering with {best_linkage} linkage and K = {optimal_hier_k}")
print(f"Silhouette Score: {silhouette_score(X_scaled, hier_labels):.4f}")
print(f"\nCluster distribution:")
unique, counts = np.unique(hier_labels, return_counts=True)
for cluster, count in zip(unique, counts):
    print(f"Cluster {cluster}: {count} samples ({count/len(hier_labels)*100:.1f}%)")

### 5.4 Visualize Hierarchical Clusters

In [ ]:
# 2D visualization of hierarchical clusters
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

# Plot all feature pairs
plot_idx = 0
for i in range(len(features_to_use)):
    for j in range(i+1, len(features_to_use)):
        if plot_idx < 6:
            scatter = axes[plot_idx].scatter(X_scaled[:, i], X_scaled[:, j], 
                                            c=hier_labels, cmap='plasma', 
                                            alpha=0.6, s=50, edgecolors='k', linewidth=0.5)
            axes[plot_idx].set_xlabel(features_to_use[i], fontsize=10)
            axes[plot_idx].set_ylabel(features_to_use[j], fontsize=10)
            axes[plot_idx].set_title(f'Hierarchical Clusters ({best_linkage})\n{features_to_use[i]} vs {features_to_use[j]}', 
                                    fontsize=11, fontweight='bold')
            axes[plot_idx].grid(True, alpha=0.3)
            plot_idx += 1

plt.colorbar(scatter, ax=axes, label='Cluster')
plt.tight_layout()
plt.show()

## 6. Comparison and Analysis

### 6.1 Compare Cluster Assignments

In [ ]:
# Create a comparison dataframe
comparison_df = pd.DataFrame({
    'K-means': kmeans_labels,
    'Hierarchical': hier_labels
})

# Cross-tabulation of cluster assignments
cross_tab = pd.crosstab(comparison_df['K-means'], comparison_df['Hierarchical'], 
                        rownames=['K-means'], colnames=['Hierarchical'])
print("Cross-tabulation of Cluster Assignments:")
print(cross_tab)
print(f"\nAgreement rate: {(kmeans_labels == hier_labels).sum() / len(kmeans_labels) * 100:.2f}%")

In [ ]:
# Visualize cluster comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# K-means
scatter1 = axes[0].scatter(X_scaled[:, 0], X_scaled[:, 1], 
                          c=kmeans_labels, cmap='viridis', 
                          alpha=0.6, s=50, edgecolors='k', linewidth=0.5)
axes[0].set_xlabel(features_to_use[0], fontsize=12)
axes[0].set_ylabel(features_to_use[1], fontsize=12)
axes[0].set_title(f'K-means Clustering (K={optimal_k})', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# Hierarchical
scatter2 = axes[1].scatter(X_scaled[:, 0], X_scaled[:, 1], 
                          c=hier_labels, cmap='plasma', 
                          alpha=0.6, s=50, edgecolors='k', linewidth=0.5)
axes[1].set_xlabel(features_to_use[0], fontsize=12)
axes[1].set_ylabel(features_to_use[1], fontsize=12)
axes[1].set_title(f'Hierarchical Clustering ({best_linkage}, K={optimal_hier_k})', 
                 fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
plt.colorbar(scatter2, ax=axes[1], label='Cluster')

plt.tight_layout()
plt.show()

### 6.2 Cluster Characteristics Analysis

In [ ]:
# Analyze cluster characteristics for K-means
df_with_clusters = df[features_to_use].copy()
df_with_clusters['KMeans_Cluster'] = kmeans_labels
df_with_clusters['Hierarchical_Cluster'] = hier_labels

print("=" * 80)
print("K-MEANS CLUSTER CHARACTERISTICS")
print("=" * 80)
print(df_with_clusters.groupby('KMeans_Cluster')[features_to_use].mean())

print("\n" + "=" * 80)
print("HIERARCHICAL CLUSTER CHARACTERISTICS")
print("=" * 80)
print(df_with_clusters.groupby('Hierarchical_Cluster')[features_to_use].mean())

In [ ]:
# Box plots to compare cluster characteristics
fig, axes = plt.subplots(2, 4, figsize=(18, 10))

for idx, feature in enumerate(features_to_use):
    # K-means
    df_with_clusters.boxplot(column=feature, by='KMeans_Cluster', ax=axes[0, idx])
    axes[0, idx].set_title(f'K-means\n{feature}')
    axes[0, idx].set_xlabel('Cluster')
    axes[0, idx].set_ylabel('Value')
    
    # Hierarchical
    df_with_clusters.boxplot(column=feature, by='Hierarchical_Cluster', ax=axes[1, idx])
    axes[1, idx].set_title(f'Hierarchical\n{feature}')
    axes[1, idx].set_xlabel('Cluster')
    axes[1, idx].set_ylabel('Value')

plt.suptitle('Cluster Characteristics Comparison', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

### 6.3 Sensitivity to Outliers

In [ ]:
# Identify potential outliers using Z-score
from scipy import stats

z_scores = np.abs(stats.zscore(X_scaled))
outliers = (z_scores > 3).any(axis=1)
n_outliers = outliers.sum()

print(f"Number of potential outliers (Z-score > 3): {n_outliers}")
print(f"Percentage of outliers: {n_outliers/len(X_scaled)*100:.2f}%")

if n_outliers > 0:
    print("\nOutlier distribution across K-means clusters:")
    outlier_kmeans = pd.Series(kmeans_labels[outliers])
    print(outlier_kmeans.value_counts())
    
    print("\nOutlier distribution across Hierarchical clusters:")
    outlier_hier = pd.Series(hier_labels[outliers])
    print(outlier_hier.value_counts())

## 7. Summary and Conclusions

In [ ]:
# Summary statistics
print("=" * 80)
print("CLUSTERING COMPARISON SUMMARY")
print("=" * 80)
print(f"\nDataset: Iris Dataset (Flower Measurements)")
print(f"Number of samples: {len(X_scaled)}")
print(f"Number of features used: {len(features_to_use)}")
print(f"Features: {', '.join(features_to_use)}")

print(f"\n{'='*40}")
print("K-MEANS CLUSTERING")
print(f"{'='*40}")
print(f"Optimal K: {optimal_k}")
print(f"Silhouette Score: {silhouette_score(X_scaled, kmeans_labels):.4f}")
print(f"Cluster sizes: {dict(zip(*np.unique(kmeans_labels, return_counts=True)))}")

print(f"\n{'='*40}")
print("HIERARCHICAL CLUSTERING")
print(f"{'='*40}")
print(f"Linkage method: {best_linkage}")
print(f"Optimal K: {optimal_hier_k}")
print(f"Silhouette Score: {silhouette_score(X_scaled, hier_labels):.4f}")
print(f"Cluster sizes: {dict(zip(*np.unique(hier_labels, return_counts=True)))}")

print(f"\n{'='*40}")
print("AGREEMENT ANALYSIS")
print(f"{'='*40}")
agreement = (kmeans_labels == hier_labels).sum() / len(kmeans_labels) * 100
print(f"Agreement rate: {agreement:.2f}%")
print("="*80)

## 8. Discussion and Insights

### Key Findings:

#### **K-means Clustering:**
- **Advantages:**
  - Faster computation, suitable for large datasets
  - Produces spherical clusters of similar sizes
  - Clear cluster separation based on centroids
  
- **Limitations:**
  - Assumes spherical clusters
  - Sensitive to initial centroid positions (though we used n_init=10)
  - May struggle with clusters of different sizes or densities

#### **Hierarchical Clustering:**
- **Advantages:**
  - No need to pre-specify number of clusters
  - Dendrogram provides insights into data structure
  - Can capture hierarchical relationships
  - Different linkage methods offer flexibility
  
- **Limitations:**
  - More computationally expensive
  - Sensitive to outliers (especially with single linkage)
  - Once merged, clusters cannot be separated

#### **When to Use Each Method:**

**Use K-means when:**
- You have large datasets
- Clusters are expected to be roughly spherical and similar in size
- Computational efficiency is important
- You have a good estimate of the number of clusters

**Use Hierarchical clustering when:**
- You want to explore hierarchical relationships in the data
- The number of clusters is unknown
- You want to visualize the clustering structure (dendrogram)
- Dataset is relatively small to medium-sized

### Limitations and Improvements:

1. **Feature Selection:** We selected 4 features; using all features or PCA might reveal different patterns
2. **Scaling:** StandardScaler was used; other scaling methods (MinMaxScaler, RobustScaler) could be tested
3. **Distance Metrics:** Euclidean distance was used; Manhattan or cosine distance might be more appropriate
4. **Validation:** Could use additional metrics like Davies-Bouldin Index or Calinski-Harabasz Score
5. **Outlier Handling:** Could remove or treat outliers before clustering
6. **Alternative Methods:** Could explore DBSCAN, Gaussian Mixture Models, or spectral clustering